# Sistemas Inteligentes — Experimento reproduzível (Semanas 1–6)

**Revisão:** 25/11/2026  
**Repositório:** https://github.com/rochelAsc/GWO-environment  
**Ambiente-alvo:** Google Colab / Python 3

Este notebook é a interface de reprodução do projeto. As implementações oficiais permanecem em scripts/; o notebook não mantém cópias paralelas dos algoritmos. Cada etapa chama os mesmos scripts versionados e protocolos usados nos experimentos auditados.

> As execuções completas podem consumir tempo considerável. As flags começam desativadas. Leia os protocolos, execute os testes e habilite explicitamente cada bateria. Cada nova sessão deve usar um identificador de saída diferente para não sobrescrever resultados anteriores.

## Escopo e honestidade experimental

| Semana | Conteúdo | Reprodução neste notebook |
|---|---|---|
| 1 | Formulação/estudo inicial do GWO; proposta original dengue + LSTM/RMSE | Registro documental. A proposta inicial não possui pipeline executável reproduzível no estado atual do repositório. |
| 2 | GWO baseline corrigido, PFSP Taillard, makespan | scripts.run_week2_benchmark |
| 3 | Sintonia Optuna e avaliação final | scripts.tune_week3_optuna + scripts.run_week3_evaluation |
| 4 | Surrogate MLP por instância | scripts.run_week4_surrogate |
| 5 | GWO com controlador Fuzzy Mamdani | scripts.run_week5_fuzzy |
| 6 | Benchmark Baseline × Optuna × Surrogate assistido × Fuzzy | scripts.run_week6_benchmark |

A formulação experimental reconstruída das Semanas 2–6 é **PFSP clássico de uma fábrica, sem due dates, minimizando makespan**, nas instâncias Taillard ta001 e ta011. Ela não deve ser confundida com os experimentos históricos de PFS/tardiness nem com DPFSP-DD. A Semana 1 teve escopo inicial diferente; essa mudança deve ser explicada na apresentação, não apagada.

### Documentos de referência

- [Auditoria Semana 1](../docs/audits/2026-11-23-week1-theoretical-review.md)
- [Protocolo Semana 2](../docs/reports/week2_reconstruction_protocol.md)
- [Protocolo Semana 3](../docs/reports/week3_optuna_protocol.md)
- [Protocolo Semana 4](../docs/reports/week4_surrogate_protocol.md)
- [Protocolo Semana 5](../docs/reports/week5_fuzzy_protocol.md)
- [Protocolo Semana 6](../docs/reports/week6_benchmark_protocol.md)
- [Auditoria Semana 6](../docs/audits/2026-11-25-week6-benchmark-audit.md)
- [Auditoria da correção do GWO](../docs/audits/2026-11-23-gwo-population-freeze-correction.md)

O notebook anterior foi preservado em Sistemas_Inteligentes_Entregavel_historico_2026-11-25.ipynb.

In [ ]:
# 1. Preparação do ambiente Colab
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/rochelAsc/GWO-environment.git"
REPO_DIR = Path("/content/GWO-environment")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Repositório já existe no runtime; mantendo a cópia local.")

os.chdir(REPO_DIR)
print("Diretório:", Path.cwd())
print(subprocess.run(["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip())

In [ ]:
# 2. Instalação das dependências
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-dev.txt"],
    check=True
)
print("Dependências instaladas.")

In [ ]:
# 3. Verificação do ambiente e dos arquivos de entrada
import hashlib
import platform
import numpy as np
import scipy
import sklearn

INSTANCE_PATHS = [
    Path("data/benchmarks/taillard/ta001.txt"),
    Path("data/benchmarks/taillard/ta011.txt"),
]
for path in INSTANCE_PATHS:
    if not path.is_file():
        raise FileNotFoundError(f"Instância ausente: {path}")
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{path}: {path.stat().st_size:,} bytes | SHA-256 {digest}")

print({
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
})

## 4. Testes automatizados

A suíte verifica parser, avaliadores, atualização da população, determinismo, orçamento de avaliações, controlador Fuzzy, Surrogate e análise estatística. Se algum teste falhar, não prossiga para as baterias.

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

## 5. Configuração das execuções

Defina um identificador novo para cada conjunto de execuções. Os resultados serão gravados em results/notebook_runs/<RUN_ID>/, sem reutilizar os diretórios oficiais já publicados.

As flags começam em False. Habilite somente a etapa que deseja executar. Para reprodução integral das Semanas 2–6, habilite-as em ordem e aguarde cada etapa terminar antes da próxima.

In [ ]:
# IMPORTANTE: altere RUN_ID a cada nova bateria.
RUN_ID = "colab_20261125_01"
RUN_ROOT = Path("results") / "notebook_runs" / RUN_ID

RUN_WEEK2_FULL = False
RUN_WEEK3_TUNING_AND_EVALUATION = False
RUN_WEEK4_SURROGATE = False
RUN_WEEK5_FUZZY = False
RUN_WEEK6_BENCHMARK = False

SEEDS = 30
WOLVES = 30
ITERATIONS = 500
WEEK4_SAMPLES_PER_INSTANCE = 2000

def run_module(module, *args):
    command = [sys.executable, "-m", module, *map(str, args)]
    print("$", " ".join(command))
    subprocess.run(command, check=True)

def require_fresh(path):
    if path.exists():
        raise FileExistsError(
            f"{path} já existe. Escolha outro RUN_ID; não sobrescreva resultados."
        )

print("Raiz das novas saídas:", RUN_ROOT)

## Semana 1 — registro teórico e mudança de escopo

A primeira proposta tratava de dengue, LSTM e RMSE, com GWO como mecanismo de otimização. Essa etapa foi documental/conceitual; não há no pipeline atual um experimento executável completo dessa formulação. A partir das reconstruções posteriores, o trabalho experimental foi consolidado em PFSP/Taillard e makespan.

**Não é correto apresentar o pipeline PFSP das Semanas 2–6 como se fosse a implementação original da Semana 1.** A apresentação deve registrar a evolução do escopo e justificar a escolha de uma formulação que passou a ter instâncias, avaliador, scripts, testes e resultados rastreáveis.

Referência: [auditoria teórica da Semana 1](../docs/audits/2026-11-23-week1-theoretical-review.md).

## Semana 2 — GWO baseline corrigido (PFSP/Taillard)

- Instâncias: ta001 e ta011.
- 30 lobos, 500 iterações, 30 seeds (0–29).
- Orçamento: 15.030 avaliações exatas por execução.
- Saídas: runs.csv, convergence.csv e metadata.json.

A versão v2 corrige o defeito de população congelada. As baterias v1 permanecem preservadas como histórico e não devem ser usadas como evidência de GWO iterativo.

In [ ]:
# Semana 2 — bateria completa (60 execuções)
week2_out = RUN_ROOT / "week2_baseline"
if RUN_WEEK2_FULL:
    require_fresh(week2_out)
    run_module(
        "scripts.run_week2_benchmark",
        "--seeds", SEEDS,
        "--wolves", WOLVES,
        "--iterations", ITERATIONS,
        "--output", week2_out,
    )
else:
    print("Ignorada. Para executar, defina RUN_WEEK2_FULL = True.")

## Semana 3 — Optuna e avaliação final

A sintonia utiliza TPE, 30 trials e seeds de treinamento 100–102. A avaliação final usa seeds 0–29, que não participam da sintonia. O orçamento por execução é mantido em 15.030 avaliações exatas.

A avaliação depende do CSV baseline gerado na Semana 2 desta mesma bateria. Não misture arquivos de execuções diferentes.

In [ ]:
# Semana 3 — tuning + avaliação final
week3_out = RUN_ROOT / "week3_optuna"
if RUN_WEEK3_TUNING_AND_EVALUATION:
    if not (week2_out / "runs.csv").is_file():
        raise FileNotFoundError("Execute a Semana 2 nesta bateria antes da Semana 3.")
    require_fresh(week3_out)
    run_module("scripts.tune_week3_optuna", "--output", week3_out)
    run_module(
        "scripts.run_week3_evaluation",
        "--config", week3_out / "best_params.json",
        "--baseline", week2_out / "runs.csv",
        "--output", week3_out,
    )
else:
    print("Ignorada. Para executar, defina RUN_WEEK3_TUNING_AND_EVALUATION = True.")

## Semana 4 — Surrogate MLP

Treina um modelo independente para cada instância, com 2.000 amostras solicitadas por instância, LHS, remoção de permutações duplicadas, divisão treino/teste e métricas de regressão. O treinamento e a geração de rótulos exatos são custos de preparação e devem permanecer separados do custo de busca.

**Atenção:** o script padrão gera um piloto de 300 amostras. Aqui especificamos 2.000 para reproduzir a configuração principal da Semana 4 usada na Semana 6.

In [ ]:
# Semana 4 — modelos necessários à Semana 6
week4_out = RUN_ROOT / "week4_surrogate"
if RUN_WEEK4_SURROGATE:
    require_fresh(week4_out)
    run_module(
        "scripts.run_week4_surrogate",
        "--samples", WEEK4_SAMPLES_PER_INSTANCE,
        "--seed", 20261123,
        "--output", week4_out,
    )
else:
    print("Ignorada. Para executar, defina RUN_WEEK4_SURROGATE = True.")

## Semana 5 — GWO com controlador Fuzzy Mamdani

O Fuzzy é executado com a mesma formulação PFSP/makespan, seeds 0–29 e orçamento de 15.030 avaliações exatas por execução. Depende dos resultados baseline e Optuna da mesma bateria, usados para comparações pareadas.

A implementação v2 atualiza a população a cada geração e registra o traço do controlador.

In [ ]:
# Semana 5 — bateria completa (60 execuções)
week5_out = RUN_ROOT / "week5_fuzzy"
if RUN_WEEK5_FUZZY:
    if not (week2_out / "runs.csv").is_file():
        raise FileNotFoundError("A Semana 5 requer o baseline da Semana 2 desta bateria.")
    if not (week3_out / "final_runs.csv").is_file():
        raise FileNotFoundError("A Semana 5 requer a avaliação Optuna da Semana 3 desta bateria.")
    require_fresh(week5_out)
    run_module(
        "scripts.run_week5_fuzzy",
        "--baseline", week2_out / "runs.csv",
        "--tuned", week3_out / "final_runs.csv",
        "--seeds", *range(SEEDS),
        "--wolves", WOLVES,
        "--budget", WOLVES * (ITERATIONS + 1),
        "--output", week5_out,
    )
else:
    print("Ignorada. Para executar, defina RUN_WEEK5_FUZZY = True.")

## Semana 6 — benchmark comparativo das quatro variantes

Compara Baseline, Optuna, Surrogate assistido e Fuzzy nas duas instâncias e 30 seeds: 240 execuções. O Surrogate usa o modelo congelado da Semana 4 e reavalia exatamente os três melhores candidatos previstos por geração.

- Baseline / Optuna / Fuzzy: 15.030 avaliações exatas por execução.
- Surrogate assistido: 1.530 avaliações exatas + 15.000 predições por execução.
- O orçamento exato não é igual entre as variantes; compare como estratégias com custos distintos.
- A bateria é incremental e pode continuar após interrupção, mas use um diretório novo para uma nova repetição.

In [ ]:
# Semana 6 — benchmark de 240 execuções
week6_out = RUN_ROOT / "week6_benchmark"
if RUN_WEEK6_BENCHMARK:
    for required in [
        week3_out / "best_params.json",
        week4_out / "ta001" / "surrogate.joblib",
        week4_out / "ta011" / "surrogate.joblib",
        week4_out / "ta001" / "metrics.json",
        week4_out / "ta011" / "metrics.json",
    ]:
        if not required.is_file():
            raise FileNotFoundError(f"Artefato necessário à Semana 6 ausente: {required}")
    require_fresh(week6_out)
    run_module(
        "scripts.run_week6_benchmark",
        "--optuna-config", week3_out / "best_params.json",
        "--surrogate-dir", week4_out,
        "--seeds", *range(SEEDS),
        "--top-k", 3,
        "--output", week6_out,
    )
else:
    print("Ignorada. Para executar, defina RUN_WEEK6_BENCHMARK = True.")

## 6. Inspeção dos artefatos gerados

As tabelas abaixo são carregadas somente se os arquivos existirem. Os resultados publicados anteriormente continuam disponíveis nas pastas results/week2_baseline_v2/, results/week3_optuna_v2/, results/week4_surrogate_v2/, results/week5_fuzzy_v2/ e results/week6_benchmark_v1/. Não confunda esses resultados auditados com uma nova execução Colab em results/notebook_runs/<RUN_ID>/.

In [ ]:
import pandas as pd
from IPython.display import display

def show_csv(path, n=10):
    path = Path(path)
    if path.is_file():
        print(path)
        display(pd.read_csv(path).head(n))
    else:
        print(f"Ainda não disponível: {path}")

show_csv(week2_out / "runs.csv")
show_csv(week3_out / "final_runs.csv")
show_csv(week5_out / "runs.csv")
show_csv(week6_out / "runs.csv")

## 7. Exportação dos resultados

O runtime do Colab é temporário. Ao terminar, compacte o diretório da bateria e baixe o ZIP ou copie-o para o Google Drive. Preserve também o SHA do commit do repositório, os metadados, os CSVs e os traces. Não publique apenas gráficos ou tabelas agregadas.

In [ ]:
import shutil
from google.colab import files

if RUN_ROOT.exists():
    archive_base = Path("/content") / f"{RUN_ID}_resultados"
    archive = shutil.make_archive(str(archive_base), "zip", root_dir=RUN_ROOT)
    print("Arquivo criado:", archive)
    files.download(archive)
else:
    print("Nenhuma saída desta bateria foi criada ainda.")

## 8. Interpretação e limites

1. A Semana 1 é registrada como histórico de escopo, não como experimento executável do pipeline PFSP.
2. As Semanas 2–6 aqui reproduzem a linha corrigida PFSP/Taillard; não reproduzem automaticamente os experimentos históricos de PFS/tardiness.
3. A bateria Semana 6 já publicada e auditada continua sendo a referência comparativa oficial. Uma reexecução Colab é uma repetição independente e deve ser analisada como tal.
4. Duas instâncias não demonstram generalização ampla. A expansão para Meta-Learning será um experimento adicional, com mais instâncias e divisão treino/teste por instância.
5. Para detalhes de parâmetros, orçamento, hashes, métricas e limitações, prevalecem os protocolos e auditorias vinculados no início deste notebook.

**Reprodução não significa apenas executar células:** significa preservar versão do código, instâncias, parâmetros, seeds, ambiente, custos e artefatos brutos.